In [1]:
pip install pandas sqlalchemy pymysql openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [3]:
DB_CONFIG={
    "user":"root",
    "password":"admin@123",
    "host":"localhost",
    "port":3306,
    "database":"hr_attrition"
}

In [5]:
encoded_password=quote_plus(DB_CONFIG["password"])
connection_string=(
    f"mysql+pymysql://{DB_CONFIG['user']}:{encoded_password}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)
engine=create_engine(connection_string)

In [12]:
import glob

folder = r"C:\Users\hp\Desktop\hr attrition"
files = glob.glob(folder + r"\*")
print(files)  # this will show you the EXACT filename with its real extension

['C:\\Users\\hp\\Desktop\\hr attrition\\WA_Fn-UseC_-HR-Employee-Attrition.csv']


In [15]:
FILE_PATH = r"C:\Users\hp\Desktop\hr attrition\WA_Fn-UseC_-HR-Employee-Attrition.csv"

print("Reading CSV file...")
df = pd.read_csv(FILE_PATH)

print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")
print("\nColumn names:")
print(df.columns.tolist())

Reading CSV file...
Loaded 1470 rows and 35 columns.

Column names:
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [16]:
df.columns=[c.strip().replace(" ","_").lower() for c in df.columns]

In [18]:
null_counts=df.isnull().sum()
if null_counts.sum()>0:
    print("\nNull values found:")
    print(null_counts[null_counts>0])
else:
    print("\nNo null values found - dataset is clean.")


No null values found - dataset is clean.


In [20]:
if "employeenumber" in df.columns:
    dupes=df.duplicated(subset=["employeenumber"]).sum()
    print(f"Duplicate employee records: {dupes}")

Duplicate employee records: 0


In [21]:
TABLE_NAME="employee_attrition"
print(f"\nWriting to MySQL table '{TABLE_NAME}'...")
df.to_sql(TABLE_NAME,con=engine,if_exists="replace",index=False,chunksize=500)
print("Done! Data loaded into MySQL.")


Writing to MySQL table 'employee_attrition'...
Done! Data loaded into MySQL.


In [22]:
with engine.connect() as conn:
    result=conn.exec_driver_sql(f"SELECT COUNT(*) FROM {TABLE_NAME}")
    count=result.fetchone()[0]
    print(f"\nVerification: {count} rows in MySQL table '{TABLE_NAME}'.")


Verification: 1470 rows in MySQL table 'employee_attrition'.
